# RikiGaku-AHIRU（AHIRU-μTJ42V）全12領域・一括物理検証システム（Google Colab Pro A100 GPU）

本ノートブックは、リポジトリの **[`index.html`](https://github.com/sc8z35a-collab/RikiGaku-AHIRU/blob/main/index.html)** から **54個の3D CADパーツ（53,127三角形・ミリ単位座標）をビット完全抽出**し、現実で製造・飛行させた際に全物理現象・衝突・熱・構造・回路・制御・形式検証・製造性を突破できるかを **ワンクリック（`ランタイム` → `すべてのセルを実行` / `Ctrl+F9`）** で一括検証する航空宇宙グレード検証システムです。

### 実行環境・ツール総動員マトリクス（Colab Pro A100 GPU 実行 + 商用CAEネイティブ出力）

| 領域 | 検証内容 | Colab Pro (A100 GPU) ライブ実行エンジン | 商用ツール互換エクスポート（ZIP同梱） |
|---:|---|---|---|
| **01** | 3D CAD・静的/動的DMU衝突・6σ公差累積 | Node.js抽出 / `trimesh` / `python-fcl` / `CadQuery` / `OpenSCAD` / **A100 10万回モンテカルロGD&T** | STEP AP214, 54 STL, OpenSCAD (NX/CATIA/Creo/SolidWorks/CETOL 6σ) |
| **02** | CFD・ターボ機械・燃焼動力学・空力 | **`Cantera`** (詳細化学反応・PSR失火) / **A100 2D圧縮性Euler CFD** / **`XFOIL`** / `OpenFOAM` | OpenFOAM (`rhoSimpleFoam`), SU2 `.cfg` (Fluent/STAR-CCM+/GasTurb/Chemkin) |
| **03** | 共役熱伝達(CHT)・熱輻射・熱暴走 | **A100 2D/3D CHT FVM** + 灰体輻射 + 停止後熱ソークバック(15分) + LiHV熱暴走 | CalculiX/Abaqus 熱 `.inp` (Fluent CHT/Icepak/COMSOL/Thermal Desktop) |
| **04** | 構造FEA・熱応力・衝撃封じ込め・破壊力学 | **`CalculiX (ccx)` 3D FEA** + `Gmsh` + 遠心バースト + ブレード飛散封じ込め + Goodman/Paris LEFM | Abaqus/CalculiX `.inp` (ANSYS Mechanical/Abaqus/LS-DYNA/nCode/FRANC3D) |
| **05** | 超高速ローター力学・軸受・空力騒音(CAA) | **`ROSS`** + 20-DOF FEジャイロCampbell線図 + Si3N4軸受DN/$L_{10h}$ + AGMA歯車 + Lighthill/FW-H CAA | Campbell/不釣り合い応答データ (ANSYS Rotordynamics/XLRotor/KISSsoft/Actran) |
| **06** | マルチボディ(MBD)・燃料スロッシング | **`MuJoCo`** 6-DOF着地衝撃接触 + **A100 3D WCSPH 4,096粒子スロッシング** + ジャイロ歳差 | MJCF `.xml` (MSC Adams/Simscape/Flow-3D/DualSPHysics/Isaac Sim) |
| **07** | 電子回路(SPICE)・基板DRC・SI/PI | **`ngspice`** 過渡回路解析 + PDNインピーダンス/伝送線路SI + **`KiCad`** 八角形FADEC PCB DRC | SPICE `.cir`, KiCad `.kicad_pcb` (Altium/LTspice/PLECS/ANSYS SIwave) |
| **08** | 電磁界(モーター/発電機)・3D FDTD EMI/EMC | 2D極座標静磁界 + 225kRPM逆起電力解析 + **A100 3D Yee-FDTD Maxwell方程式 (CISPR-25)** | FEMM `.lua` (ANSYS Maxwell/JMAG/ANSYS HFSS/CST Studio/openEMS) |
| **09** | 飛行制御(GNC)・6自由度非線形SIL | **`python-control`** + **`CasADi`** NMPC + CARE MIMO LQR + 6-DOF 突風/スロッシング/巡航遷移SIL | JSBSim `.xml` (MATLAB Simulink/PX4-Gazebo/dSPACE/Speedgoat HIL) |
| **10** | 組み込みFW・静的解析・WCET・形式検証 | C11 FADEC FW生成 + **`Cppcheck`** + **`Clang Static Analyzer`** + ASan/UBSan WCET + **`SPIN`** + **`Z3`** | C11 `.c/.h`, Promela `.pml` (Polyspace/Coverity/AbsInt aiT/TLA+/UPPAAL) |
| **11** | システム設計(MBSE)・FTA・FMEA (RAMS) | **`SCRAM`** Open-PSA故障木解析(最小カットセット) + 物理連動FMEA + SysML RTM/ICD監査 | Open-PSA `.xml`, SysML RTM `.json` (Cameo/Capella/ReliaSoft/Isograph) |
| **12** | 製造性(DFM/DFA)・SLM金属3D造形・5軸CNC | 固有ひずみSLM残留歪み/オーバーハング + **A100 5軸CNC干渉レイキャスト** + 型抜き勾配/DFA | DFM/CAM `.json` (ANSYS Additive/Moldflow/VERICUT/Mastercam/NX CAM) |

> **重要（誠実な物理検証の原則）**: 商用有償ライセンス必須ツール（NX/CATIA/ANSYS/Abaqus/Altium/MATLAB/VERICUT等）はGoogle Colab上で直接起動できないため、本システムは **すべての領域で同等のオープンソース物理エンジン＋A100 GPU直接数値解法を実行** しつつ、商用ツールへそのままインポート可能なネイティブ入力デッキを全自動生成します。また合格判定を甘くごまかすことは一切せず、実機コピー時に問題となる箇所を **厳格に `FAIL` / `MARGINAL` として定量診断** します。


In [ ]:
#@title 1. ワンクリック設定 & NVIDIA A100 GPU・リポジトリ同期 { display-mode: "form" }
REPO_URL = "https://github.com/sc8z35a-collab/RikiGaku-AHIRU.git" #@param {type:"string"}
BRANCH = "main" #@param {type:"string"}
INSTALL_HEAVY_APT = False #@param {type:"boolean"}
AUTO_DOWNLOAD_ZIP = True #@param {type:"boolean"}

import os, subprocess, sys
from pathlib import Path

print("=== [1/5] Checking NVIDIA GPU / CUDA Environment ===")
subprocess.run(["nvidia-smi"], check=False)
try:
    import torch
    if torch.cuda.is_available():
        props = torch.cuda.get_device_properties(0)
        print(f"PyTorch CUDA Ready: {props.name} | VRAM: {props.total_memory / 1024**3:.1f} GB | CUDA {torch.version.cuda}")
    else:
        print("WARNING: GPU runtime not active. Switch Runtime -> Change runtime type -> A100 GPU for maximum speed!")
except ImportError:
    print("PyTorch not preinstalled; will install in Step 2.")

REPO_DIR = Path("/content/RikiGaku-AHIRU")
if REPO_DIR.exists():
    print(f"Updating existing repository in {REPO_DIR}...")
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "reset", "--hard", f"origin/{BRANCH}"], check=True)
else:
    print(f"Cloning {REPO_URL} (branch={BRANCH})...")
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
print(f"Working directory ready: {os.getcwd()}")


In [ ]:
#@title 2. 全12領域オープンソース物理エンジン一括インストール（apt + pip）
import subprocess, sys

print("=== [2/5] Installing Native Linux Aerospace Solvers (apt) ===")
core_apt = [
    "nodejs",
    "calculix-ccx",
    "gmsh",
    "ngspice",
    "xfoil",
    "openscad",
    "cppcheck",
    "clang",
    "spin",
    "scram",
    "libfcl-dev",
    "liboctomap-dev",
]
if INSTALL_HEAVY_APT:
    core_apt += ["kicad", "openfoam"]

subprocess.run(["apt-get", "update", "-qq"], check=False)
for pkg in core_apt:
    r = subprocess.run(["apt-get", "install", "-y", "-qq", pkg], capture_output=True, text=True)
    status = "OK" if r.returncode == 0 else "SKIP"
    print(f"  apt [{status:4s}] {pkg}")

print("\n=== Installing Python Aerospace & Multiphysics Packages (pip) ===")
pip_pkgs = [
    "trimesh",
    "python-fcl",
    "cantera",
    "ross-rotordynamics",
    "mujoco",
    "control",
    "casadi",
    "z3-solver",
    "scipy",
    "matplotlib",
]
for pkg in pip_pkgs:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], capture_output=True, text=True)
    status = "OK" if r.returncode == 0 else "WARN"
    print(f"  pip [{status:4s}] {pkg}")

# Optional CadQuery (ISO STEP AP214 B-Rep export)
r_cq = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "cadquery"], capture_output=True, text=True)
print(f"  pip [{'OK' if r_cq.returncode == 0 else 'OPT ':4s}] cadquery (STEP AP214 B-Rep)")

print("\nToolchain installation complete!")


In [ ]:
#@title 3. 全12領域・一括物理検証の実行（NVIDIA A100 GPU フル稼働）
from pathlib import Path
from verification.run_all import run_full_verification

OUT_DIR = Path("/content/ahiru_v12_out")
cad, reports, outputs = run_full_verification(
    html_path="index.html",
    out_dir=OUT_DIR,
    verbose=True,
)


In [ ]:
#@title 4. エグゼクティブ検証ダッシュボード & 全12領域エンジニアリング解析図の表示
from IPython.display import HTML, Image, display
import json

print("=== [4/5] Executive Scorecard Summary ===")
data = json.loads(outputs["json"].read_text(encoding="utf-8"))
s = data["summary"]
print(f"Overall Verdict : {data['overall_verdict']}")
print(f"Checks Summary  : {s['checks_pass']} PASS | {s['checks_marginal']} MARGINAL | {s['checks_fail']} FAIL (Total {s['checks_total']} checks, {s['pass_rate_pct']}%)")
print(f"Domains 100% OK : {s['domains_fully_passed']} / {s['domains_total']}")
print("\n--- Domain-by-Domain Breakdown ---")
for d in data["domains"]:
    print(f"  Domain {d['domain_id']:02d} [{d['overall_verdict']:8s}] {d['title_ja']} ({d['pass_count']}P / {d['marginal_count']}M / {d['fail_count']}F, {d['elapsed_sec']:.2f}s)")

# Render complete self-contained HTML verification dashboard directly in Colab!
display(HTML(outputs["html"].read_text(encoding="utf-8")))


In [ ]:
#@title 5. 全検証レポート + 商用CAE互換デッキ一式（ZIP）のワンクリックダウンロード
import os
from pathlib import Path

zip_path = outputs["zip"]
size_mb = zip_path.stat().st_size / (1024 ** 2)
print(f"=== [5/5] Verification Archive Ready: {zip_path} ({size_mb:.2f} MB) ===")
print("\nIncluded Commercial CAE Interchange Decks & Reports:")
for p in sorted(OUT_DIR.rglob("*")):
    if p.is_file() and p.suffix in (".html", ".json", ".md", ".step", ".scad", ".inp", ".cfg", ".xml", ".cir", ".kicad_pcb", ".lua", ".c", ".h", ".pml"):
        print(f"  - {p.relative_to(OUT_DIR)}")

if AUTO_DOWNLOAD_ZIP:
    try:
        from google.colab import files
        files.download(str(zip_path))
    except ImportError:
        print("(Not running inside Google Colab UI — archive saved locally at:", zip_path, ")")
